<a href="https://colab.research.google.com/github/OMSCGR/Senderos-Seguros-Evaluacion-de-Impacto/blob/main/senderos_evaluacion_impacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación de impacto — Senderos Seguros
### Notebook reproducible del método descrito en la nota técnica *"¿Funcionan los Senderos Seguros?"*

Este notebook reproduce, de principio a fin, el método de la sección 3 del documento: construcción de un contrafactual agregado (10 senderos, cohorte 2024) ajustado por un efecto de contexto estimado con corredores del programa aún no intervenidos, y reproduce las tres tablas del documento:

- **Tabla 1** — resultado agregado (sección 4.1)
- **Tabla 2** — prueba de sensibilidad con y sin el corredor atípico (sección 4.2)
- **Tabla 3** — detalle por sendero individual (Anexo)

**Entrada:** `senderos_150m.csv` (subconjunto a 150 m ya filtrado del archivo original `260618_Base_indicentes_delitos_en_senderos_2023_2026.xlsx` de la Secretaría).


## 1. Cargar los datos

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats as sstats

df150 = pd.read_csv('senderos 150m.csv', parse_dates=['Fecha'])
df150['SENDERO'] = df150['SENDERO'].str.strip()
print(df150.shape)
df150.head()

(30681, 19)


,SENDERO,AÑO_ENTRE,BUFF_DIST,Longitud,Latitud,CATEGORIA,SUBCATEG_1,Fecha,AÑO,MES,DIA,NUM_DIA,RANGO_DE_H,dpa_barrio,Nombre_Bar,Cod_Barrio,Parroquia,Cod_Parroq,Admin_Zona
0,CALLE S56A,2025,150,-78.562854,-0.336858,Convivencia ciudadana,Escándalos,2025-01-21,2025,Ene,martes,21,21:00 a 23:59,170112026,SAN FERNANDO,QUI_170112026,GUAMANÍ,170112,QUITUMBE
1,CALLE S56A,2025,150,-78.562854,-0.336858,Convivencia ciudadana,Libadores,2026-05-10,2026,May,domingo,10,21:00 a 23:59,170112026,SAN FERNANDO,QUI_170112026,GUAMANÍ,170112,QUITUMBE
2,CALLE S56A,2025,150,-78.562438,-0.336689,Convivencia ciudadana,Venta y consumo de sustancias sujetas a fiscal...,2025-08-20,2025,Ago,miércoles,20,18:00 a 20:59,170112026,SAN FERNANDO,QUI_170112026,GUAMANÍ,170112,QUITUMBE
3,CALLE S56A,2025,150,-78.562001,-0.336543,Convivencia ciudadana,Libadores,2025-02-09,2025,Feb,domingo,9,21:00 a 23:59,170112026,SAN FERNANDO,QUI_170112026,GUAMANÍ,170112,QUITUMBE
4,CALLE S56A,2025,150,-78.562001,-0.336543,Convivencia ciudadana,Escándalos,2024-09-07,2024,Sep,sábado,7,03:00 a 05:59,170112026,SAN FERNANDO,QUI_170112026,GUAMANÍ,170112,QUITUMBE


## 2. Cohorte evaluable y parámetros del ejercicio

Solo los 10 senderos inaugurados en 2024 cuentan con al menos un corredor del programa que permaneciera "aún no intervenido" durante toda su ventana de seguimiento (sección 2 y 3 del documento). `CUTOFF` es la fecha de corte de la base; `W` es el número de semanas antes/después (52 = un año).

In [3]:
CUTOFF = pd.Timestamp('2026-06-14')
W = 52  # semanas antes y después (1 año)

# fecha de inauguración de los 19 senderos del programa (necesarias las 19: las 9 de
# 2025 se usan como posibles corredores de referencia/donantes para los de 2024)
treat_dates = {
    'AV PATRIA': '2024-01-25', 'EL TINGO': '2024-02-23', 'LA MICHELENA': '2024-04-25',
    'CALDAS Y ANTEPARA': '2024-05-15', 'CHILLOGALLO': '2024-06-12', 'ISLA TORTUGA': '2024-10-01',
    'LA ROLDOS': '2024-10-01', 'AV CARAPUNGO': '2024-10-09', 'AV COLÓN': '2024-12-04', 'TUMBACO': '2024-12-08',
    'AV. AMAZONAS': '2025-07-25', 'GABRIEL GARCÍA MORENO': '2025-10-22', 'CALLE ROCAFUERTE': '2025-12-15',
    'CALLE SIMÓN BOLIVAR': '2025-12-15', 'RUIZ DE CASTILLA': '2025-12-16', 'AV. LIZARDO RUIZ': '2025-12-17',
    'CALLE S56A': '2025-12-19', 'AV. AJAVÍ': '2025-12-20', 'VIA DEL FERROCARRIL': '2025-12-26',
}
treat_dates = {k: pd.Timestamp(v) for k, v in treat_dates.items()}

# cohorte evaluable (sección 2 del documento): los 10 senderos de 2024
anual_list = ['AV PATRIA', 'EL TINGO', 'LA MICHELENA', 'CALDAS Y ANTEPARA', 'CHILLOGALLO',
              'ISLA TORTUGA', 'LA ROLDOS', 'AV CARAPUNGO', 'AV COLÓN', 'TUMBACO']

CATEGORIAS = ['Convivencia ciudadana', 'ROBOS DE MAYOR CONNOTACIÓN']


## 3. Funciones auxiliares

`build_daily_lookup` precalcula, una sola vez, el conteo diario acumulado de cada sendero y categoría (usando una suma acumulada para que cualquier conteo semanal posterior sea una simple resta, en vez de repetir un filtro sobre toda la base cada vez). `weekly_counts` consulta ese precálculo.

In [4]:
FULL_START = pd.Timestamp('2022-01-01')
FULL_END = pd.Timestamp('2026-12-31')
_ALL_DAYS = pd.date_range(FULL_START, FULL_END, freq='D')

def build_daily_lookup(df150):
    """Precalcula, por (categoría, sendero), la suma acumulada diaria de incidentes.
    Permite que cualquier conteo semanal se obtenga con una resta en vez de un filtro."""
    lookup = {}
    for cat in df150['CATEGORIA'].unique():
        for sendero in df150['SENDERO'].unique():
            sub = df150[(df150['CATEGORIA'] == cat) & (df150['SENDERO'] == sendero)]
            daily = sub.groupby(sub['Fecha'].dt.floor('D')).size().reindex(_ALL_DAYS, fill_value=0)
            lookup[(cat, sendero)] = np.concatenate([[0], np.cumsum(daily.values)])
    return lookup


def weekly_counts(lookup, cat, sendero, start_date, n_weeks):
    """Conteo de incidentes por semana (n_weeks bloques de 7 días desde start_date)."""
    cum = lookup[(cat, sendero)]
    day0 = (start_date - FULL_START).days
    idx = day0 + 7 * np.arange(n_weeks + 1)
    return np.diff(cum[idx])


def fit_and_forecast(y_pre, n_steps):
    """Apartado 3.2: regresión lineal de ln(1+incidentes) sobre el número de semana,
    ajustada en el periodo previo y proyectada n_steps semanas hacia adelante.
    Devuelve la media proyectada y su intervalo de predicción al 95%."""
    N = len(y_pre)
    w_pre = np.arange(-N, 0)
    X = sm.add_constant(w_pre)
    model = sm.OLS(y_pre, X).fit()
    w_post = np.arange(0, n_steps)
    Xpred = sm.add_constant(w_post, has_constant='add')
    pred = model.get_prediction(Xpred)
    return pred.predicted_mean, pred.conf_int(obs=True, alpha=0.05)


print('Precalculando conteos diarios (una sola vez)...')
LOOKUP = build_daily_lookup(df150)
print('Listo:', len(LOOKUP), 'combinaciones categoría-sendero')


Precalculando conteos diarios (una sola vez)...
Listo: 57 combinaciones categoría-sendero


## 4. Contrafactual agregado (sección 3 del documento)

Implementa, en una sola función, los apartados 3.1 a 3.4:
1. Alinea y agrega (suma) los senderos de `unit_list` semana a semana (relativa a su propia inauguración).
2. Ajusta una única tendencia sobre la serie agregada previa y la proyecta (contrafactual propio).
3. Construye la serie agregada de corredores de referencia ("donantes": aún no intervenidos durante toda la ventana de cada sendero) y calcula su propia desviación frente a su propia tendencia — el efecto de contexto.
4. Ajusta el contrafactual propio con el efecto de contexto (piso en 0) y prueba significancia con una prueba t de una muestra sobre los residuos estandarizados.

In [5]:
def aggregate_contrafactual(lookup, cat, unit_list, treat_dates, W=52):
    pool_pre = np.zeros(W); pool_post = np.zeros(W)
    donor_pre = np.zeros(W); donor_post = np.zeros(W)

    for sendero in unit_list:
        inaug = treat_dates[sendero]
        pool_pre += weekly_counts(lookup, cat, sendero, inaug - pd.Timedelta(weeks=W), W)
        pool_post += weekly_counts(lookup, cat, sendero, inaug, W)

        post_end = inaug + pd.Timedelta(weeks=W)
        donors = [s for s in treat_dates if s != sendero and treat_dates[s] > post_end]
        for d in donors:
            donor_pre += weekly_counts(lookup, cat, d, inaug - pd.Timedelta(weeks=W), W)
            donor_post += weekly_counts(lookup, cat, d, inaug, W)

    # 3.2 — proyección propia de la serie agregada tratada
    mu_own, ci_own = fit_and_forecast(np.log1p(pool_pre), W)
    # 3.3 — efecto de contexto: desviación de la serie de referencia frente a SU PROPIA tendencia
    mu_donor, _ = fit_and_forecast(np.log1p(donor_pre), W)
    city_effect = np.log1p(donor_post) - mu_donor
    # 3.4 — contrafactual ajustado (piso en 0) y prueba de significancia
    mu_adj = mu_own + city_effect
    proyectado = np.clip(np.expm1(mu_adj), 0, None)

    y_actual = np.log1p(pool_post)
    se = np.clip((ci_own[:, 1] - ci_own[:, 0]) / (2 * 1.96), 1e-6, None)
    z = (y_actual - mu_adj) / se
    _, pval = sstats.ttest_1samp(z, 0)

    return {
        'real': pool_post.sum(),
        'proyectado': proyectado.sum(),
        'diferencia': pool_post.sum() - proyectado.sum(),
        'p_valor': pval,
        'gap_semanal': pool_post - proyectado,  # serie semanal, útil para graficar la trayectoria
    }


## 5. Tabla 1 — Resultado agregado (sección 4.1)

In [6]:
tabla1 = []
for cat in CATEGORIAS:
    r = aggregate_contrafactual(LOOKUP, cat, anual_list, treat_dates)
    tabla1.append([cat, round(r['real'], 1), round(r['proyectado'], 1),
                    round(r['diferencia'], 1), round(r['p_valor'], 4)])

tabla1 = pd.DataFrame(tabla1, columns=['Categoría', 'Real (52 sem. post)', 'Contrafactual ajustado', 'Diferencia', 'p-valor'])
tabla1


,Categoría,Real (52 sem. post),Contrafactual ajustado,Diferencia,p-valor
0,Convivencia ciudadana,3001.0,3381.2,-380.2,0.0000
1,ROBOS DE MAYOR CONNOTACIÓN,444.0,539.0,-95.0,0.0005


## 6. Tabla 2 — Prueba de sensibilidad por corredor (sección 4.2)

Se recalcula el contrafactual agregado excluyendo, uno a la vez, cada uno de los 10 senderos (barrido *leave-one-out*), para identificar si el resultado depende de un caso individual. La celda siguiente reproduce ese barrido completo; la tabla final del documento solo reporta el caso más influyente (Av. Carapungo — es el que más cambia el resultado agregado al excluirlo).

In [7]:
leave_one_out = []
for excluido in anual_list:
    resto = [s for s in anual_list if s != excluido]
    for cat in CATEGORIAS:
        r = aggregate_contrafactual(LOOKUP, cat, resto, treat_dates)
        leave_one_out.append([excluido, cat, round(r['diferencia'], 1), round(r['p_valor'], 4)])

leave_one_out = pd.DataFrame(leave_one_out, columns=['Sendero excluido', 'Categoría', 'Diferencia (9 senderos)', 'p-valor'])
leave_one_out.sort_values(['Categoría', 'Diferencia (9 senderos)'])


,Sendero excluido,Categoría,Diferencia (9 senderos),p-valor
4,LA MICHELENA,Convivencia ciudadana,-676.2,0.0000
0,AV PATRIA,Convivencia ciudadana,-516.5,0.0000
2,EL TINGO,Convivencia ciudadana,-453.5,0.0000
6,CALDAS Y ANTEPARA,Convivencia ciudadana,-385.1,0.0000
18,TUMBACO,Convivencia ciudadana,-360.4,0.0000
10,ISLA TORTUGA,Convivencia ciudadana,-314.2,0.0000
8,CHILLOGALLO,Convivencia ciudadana,-285.1,0.0000
12,LA ROLDOS,Convivencia ciudadana,-222.3,0.0001
16,AV COLÓN,Convivencia ciudadana,-208.3,0.0001
14,AV CARAPUNGO,Convivencia ciudadana,10.4,0.8286


In [8]:
# Tabla 2 del documento: 10 senderos vs. 9 (excluyendo el corredor identificado como atípico)
sin_carapungo = [s for s in anual_list if s != 'AV CARAPUNGO']

tabla2 = []
for cat in CATEGORIAS:
    r10 = aggregate_contrafactual(LOOKUP, cat, anual_list, treat_dates)
    r9 = aggregate_contrafactual(LOOKUP, cat, sin_carapungo, treat_dates)
    tabla2.append([
        cat,
        f"{r10['diferencia']:.1f} (p={r10['p_valor']:.4f})",
        f"{r9['diferencia']:.1f} (p={r9['p_valor']:.4f})",
    ])

tabla2 = pd.DataFrame(tabla2, columns=['Categoría', '10 senderos', '9 senderos (sin Av. Carapungo)'])
tabla2


,Categoría,10 senderos,9 senderos (sin Av. Carapungo)
0,Convivencia ciudadana,-380.2 (p=0.0000),10.4 (p=0.8286)
1,ROBOS DE MAYOR CONNOTACIÓN,-95.0 (p=0.0005),-100.7 (p=0.0003)


## 7. Tabla 3 — Detalle por sendero (Anexo)

Misma lógica de contrafactual (proyección propia ajustada por efecto de contexto), aplicada unidad por unidad en lugar de sobre la serie agregada.

In [9]:
def per_sendero_table(lookup, cat, unit_list, treat_dates, W=52):
    rows = []
    for sendero in unit_list:
        inaug = treat_dates[sendero]
        y_pre_t = np.log1p(weekly_counts(lookup, cat, sendero, inaug - pd.Timedelta(weeks=W), W))
        mu_t, ci_t = fit_and_forecast(y_pre_t, W)
        actual_t = weekly_counts(lookup, cat, sendero, inaug, W)

        post_end = inaug + pd.Timedelta(weeks=W)
        donors = [s for s in treat_dates if s != sendero and treat_dates[s] > post_end]
        surprises = []
        for d in donors:
            y_pre_d = np.log1p(weekly_counts(lookup, cat, d, inaug - pd.Timedelta(weeks=W), W))
            mu_d, _ = fit_and_forecast(y_pre_d, W)
            actual_d = weekly_counts(lookup, cat, d, inaug, W)
            surprises.append(np.mean(np.log1p(actual_d) - mu_d))
        city_effect = np.mean(surprises)

        mu_adj = mu_t + city_effect
        proyectado = np.clip(np.expm1(mu_adj), 0, None)
        diferencia = actual_t.sum() - proyectado.sum()

        y_actual = np.log1p(actual_t)
        se = np.clip((ci_t[:, 1] - ci_t[:, 0]) / (2 * 1.96), 1e-6, None)
        z = (y_actual - mu_adj) / se
        _, pval = sstats.ttest_1samp(z, 0)

        rows.append([sendero.title(), len(donors), round(diferencia, 1), round(pval, 3)])

    return pd.DataFrame(rows, columns=['Sendero', 'Donantes', 'Diferencia', 'p-valor'])


tabla3_conv = per_sendero_table(LOOKUP, 'Convivencia ciudadana', anual_list, treat_dates)
tabla3_conv.insert(1, 'Categoría', 'Convivencia')
tabla3_robos = per_sendero_table(LOOKUP, 'ROBOS DE MAYOR CONNOTACIÓN', anual_list, treat_dates)
tabla3_robos.insert(1, 'Categoría', 'Robos')

tabla3 = pd.concat([tabla3_conv, tabla3_robos], ignore_index=True)
tabla3


,Sendero,Categoría,Donantes,Diferencia,p-valor
0,Av Patria,Convivencia,9,72.1,0.018
1,El Tingo,Convivencia,9,59.6,0.000
2,La Michelena,Convivencia,9,272.8,0.000
3,Caldas Y Antepara,Convivencia,9,80.8,0.073
4,Chillogallo,Convivencia,9,-147.1,0.000
5,Isla Tortuga,Convivencia,8,-16.0,0.003
6,La Roldos,Convivencia,8,-119.5,0.000
7,Av Carapungo,Convivencia,8,-243.9,0.000
8,Av Colón,Convivencia,7,-318.7,0.000
9,Tumbaco,Convivencia,7,37.8,0.416


## 8. Exportar resultados

In [10]:
tabla1.to_csv('tabla1_resultado_agregado.csv', index=False)
tabla2.to_csv('tabla2_sensibilidad.csv', index=False)
tabla3.to_csv('tabla3_detalle_por_sendero.csv', index=False)
leave_one_out.to_csv('anexo_leave_one_out_completo.csv', index=False)

print('Tablas exportadas: tabla1_resultado_agregado.csv, tabla2_sensibilidad.csv, tabla3_detalle_por_sendero.csv, anexo_leave_one_out_completo.csv')


Tablas exportadas: tabla1_resultado_agregado.csv, tabla2_sensibilidad.csv, tabla3_detalle_por_sendero.csv, anexo_leave_one_out_completo.csv
